# 02 — Bounded respiratory state occurrences

We now create one object per consecutive directional state. These occurrences
are not automatically breaths; the full BIDMC construction adds
plateau-aware trough–peak–trough identity.


In [ ]:
import numpy as np
import pandas as pd
import featuregraph as fg
from featuregraph.operators.states import rising_state

signal = np.array([0, .5, 1, 1, .5, 0, 0, .6, 1.2, .6, 0, 0])
observations = pd.DataFrame({"sample": np.arange(len(signal)), "signal": signal})
observations["rate"] = observations["signal"].diff().fillna(0.0)
rate = {"column": "rate"}; eps = {"parameter": "eps"}
contract = {
    "version": "state-contract-v1", "parameters": {"eps": 1e-12},
    "states": {
        "rising": {"op": "gt", "left": rate, "right": eps},
        "falling": {"op": "lt", "left": rate,
                    "right": {"op": "neg", "value": eps}},
        "inactive": {"op": "le", "left": {"op": "abs", "value": rate},
                     "right": eps},
    },
    "events": {"enter_state": {"type": "enter_label"},
               "exit_state": {"type": "exit_label"}},
}
compiled = fg.compile_states(observations, contract)


In [ ]:
check = fg.transition.Transition(
    observations.copy(), "signal", "rising", rising_state, eps=1e-12
)
assert check.df.loc[check.df["signal_rising"], "sample"].tolist() == [1, 2, 7, 8]


In [ ]:
occurrences = fg.from_state_sequence(
    compiled.observations["state"],
    signal=compiled.observations["signal"],
    times=compiled.observations["sample"],
    group_id="tutorial-subject",
    dataset="bidmc-shaped-respiration-tutorial",
    signal_name="respiration",
    detector="state-contract-v1",
    software_version=fg.__version__,
)
objects = occurrences.object_table()
objects[["object_id", "state_label", "status", "start_index", "end_index",
         "duration", "sample_count", "signal_minimum", "signal_maximum"]]


In [ ]:
occurrences.relations


In [ ]:
rising_objects = objects.loc[
    objects["state_label"].eq("rising"),
    ["object_id", "start_index", "end_index", "duration",
     "signal_minimum", "signal_maximum"],
]
rising_objects


In [ ]:
assert objects["state_label"].tolist() == [
    "inactive", "rising", "inactive", "falling",
    "inactive", "rising", "falling", "inactive",
]
assert objects.iloc[0]["status"] == "boundary_truncated"
assert objects.iloc[-1]["status"] == "boundary_truncated"
assert objects.iloc[1:-1]["status"].eq("complete").all()
assert np.array_equal(
    occurrences.reconstruct_states(), compiled.observations["state"].to_numpy()
)
assert len(occurrences.relations) == len(objects) - 1


Once identity exists, pandas queries objects without
detecting boundaries again. The maintained BIDMC workflow adds the scientific
and comparison layers deliberately omitted here.
